#

# 📊 NOTEBOOK 1: COLETA E PREPARAÇÃO DE DADOS
**Análise de Sentimento em Decisões Judiciais - TJCE**

## 🔧 1. CONFIGURAÇÃO DO AMBIENTE

### 1.1 Instalação de Dependências
**Descrição**: Instalação das bibliotecas Python necessárias.

In [3]:
# Instalar bibliotecas necessárias
!pip install -r requirements.txt

### 1.2 Importação de Bibliotecas

In [4]:
import pandas as pd
import json
from collections import Counter
import requests
import csv

## 📥 2. COLETA DE DADOS VIA API CNJ *DATAJUD*


### 2.1 Configuração da API e Parâmetros de Busca

 **Descrição**: Configuração da requisição à API pública do CNJ. O código de assunto 12487 refere-se a "Fornecimento de medicamentos", encontrado no site oficial do CNJ (https://www.cnj.jus.br/sgt/consulta_publica_assuntos.php).

 Para encontrar o código no site do cnj:
   1. Procure por "Medicamentos"
   2. Selecione a 4ª opção: `Fornecimento de medicamentos`
   3. Verifique o código que aparece, `12487 Fornecimento de medicamentos`

In [ ]:
url = "https://api-publica.datajud.cnj.jus.br/api_publica_tjma/_search"
api_key = "APIKey cDZHYzlZa0JadVREZDJCendQbXY6SkJlTzNjLV9TRENyQk1RdnFKZGRQdw=="

payload = json.dumps({
    "size": 10000,
    "query": {"match": {"assuntos.codigo": "12487"}},  # Código: Fornecimento de medicamentos
    "sort": [{"dataAjuizamento": {"order": "desc"}}]
})

headers = {
    'Authorization': api_key,
    'Content-Type': 'application/json'
}

### 2.2 Requisição e Validação dos Dados
**Descrição**: Execução da requisição HTTP POST para a API e validação inicial do volume de dados retornados.

In [6]:
response = requests.request("POST", url, headers=headers, data=payload)
dados_dict = response.json()
print(f"Total de processos encontrados na API: {dados_dict['hits']['total']['value']}")
print(f"Processos retornados nesta consulta: {len(dados_dict['hits']['hits'])}")

Total de processos encontrados na API: 1092
Processos retornados nesta consulta: 1092


## 🔄 3. PROCESSAMENTO E ESTRUTURAÇÃO DOS DADOS

### 3.1 Extração de Campos Essenciais
**Descrição**: Extração dos campos essenciais de cada processo (número, grau de jurisdição, data de ajuizamento e movimentos processuais) e criação do DataFrame principal.

In [7]:
processos = []

for hit in dados_dict['hits']['hits']:
    processo = hit['_source']

    numero_processo = processo['numeroProcesso']
    grau = processo['grau']
    data_ajuizamento = processo['dataAjuizamento']
    movimentos = processo.get('movimentos', [])

    processos.append([
        numero_processo,
        grau,
        data_ajuizamento,
        movimentos
    ])

df = pd.DataFrame(
    processos,
    columns=[
        'numero_processo',
        'grau',
        'data_ajuizamento',
        'movimentos'
    ]
)

print(f"Total de processos no DataFrame: {len(df)}")
df.sample(5)

Total de processos no DataFrame: 1092


,numero_processo,grau,data_ajuizamento,movimentos
1009,00333586520198270000,G2,20191118100143,[]
890,00165992220208272706,G1,20200716100011,"[{'complementosTabelados': [{'codigo': 2, 'val..."
799,00216199120208272706,G1,20201022120750,"[{'complementosTabelados': [{'codigo': 2, 'val..."
791,00403540620208272729,G1,20201028143525,"[{'complementosTabelados': [{'codigo': 2, 'val..."
125,00017147320248272702,G2,20250617123541,"[{'complementosTabelados': [{'codigo': 2, 'val..."


## 🔍 4. IDENTIFICAÇÃO E ANÁLISE DE DECISÕES

### 4.1 Definição de Termos de Decisão
**Descrição**: Definição dos termos-chave que caracterizam decisões judiciais relevantes para a análise.

In [8]:
from collections import Counter

termos_decisao = [
    "Procedência",
    "Improcedência",
    "Improcedência do pedido e improcedência do pedido contraposto",
    "Procedência do pedido e procedência do pedido contraposto",
    "Deferido",
    "Indeferido",
]

### 4.2 Extração de Decisões dos Movimentos Processuais
**Descrição**: Busca sistemática nos movimentos processuais por termos de decisão.

**Importante**: Filtra apenas processos de primeira instância (grau G1) para garantir homogeneidade na análise.

In [9]:
decisoes_por_processo = []
tipos_decisao_contagem = []

for _, row in df.iterrows():
    numero = row['numero_processo']
    movimentos = row['movimentos']
    grau = row['grau']
    data_ajuizamento = row['data_ajuizamento']

    decisoes_encontradas = []

    if movimentos:
        for mov in movimentos:
            nome_mov = mov.get('nome', '')

            # Filtrar apenas decisões de primeira instância (G1)
            if any(termo in nome_mov for termo in termos_decisao) and grau == 'G1':
                decisoes_encontradas.append(nome_mov)
                tipos_decisao_contagem.append(nome_mov)

    if decisoes_encontradas:
        decisoes_por_processo.append({
            'numero_processo': numero,
            'data_ajuizamento': data_ajuizamento,
            'decisoes': decisoes_encontradas
        })

print(f"Processos com decisões: {len(decisoes_por_processo)} de {len(df)}")
print("\nTipos de decisões encontradas:")
for tipo, count in Counter(tipos_decisao_contagem).most_common(10):
    print(f"  {tipo}: {count}")

Processos com decisões: 253 de 1092

Tipos de decisões encontradas:
  Procedência: 191
  Improcedência: 36
  Procedência em Parte: 27
  Procedência do Pedido - Reconhecimento pelo réu: 1


### 4.3 Criação de DataFrame de Decisões
**Descrição**: Criação de um DataFrame específico para decisões, removendo duplicatas por processo (mantém apenas a primeira decisão encontrada).

In [10]:
decisoes_lista = []

for item in decisoes_por_processo:
    for decisao in item['decisoes']:
        decisoes_lista.append({
            'numero_processo': item['numero_processo'],
            'tipo_decisao': decisao,
        })

df_decisoes = pd.DataFrame(decisoes_lista)
df_decisoes = df_decisoes.drop_duplicates(subset='numero_processo', keep='first')
df_decisoes.head(10)

,numero_processo,tipo_decisao
0,00479211520258272729,Procedência em Parte
1,00276923420258272729,Procedência em Parte
2,00026203920258272731,Improcedência
3,00159934620258272729,Procedência em Parte
4,00159415020258272729,Procedência em Parte
5,00064031720258272706,Procedência em Parte
6,00006577520258272737,Procedência
7,00075235420248272731,Procedência
8,00018117320248272702,Improcedência
9,00073607420248272731,Improcedência


## 📊 5. SEGMENTAÇÃO POR TIPO DE DECISÃO

### 5.1 Separação por Categorias de Decisão
**Descrição**: Segmentação do dataset em categorias de decisão para análise comparativa posterior.

In [17]:
# Separar decisões por tipo
df_procedencia = df_decisoes[df_decisoes['tipo_decisao'] == 'Procedência'].copy()
df_improcedencia = df_decisoes[df_decisoes['tipo_decisao'] == 'Improcedência'].copy()

df_decisoes_completo = pd.concat([
    df_procedencia,
    df_improcedencia
])

print(f"\n=== TODOS OS REGISTROS COM DECISÃO ===")
print(f"Total de registros: {len(df_decisoes_completo)}")
print(f"\nDistribuição por tipo:")
print(f"  - Procedência: {len(df_procedencia)}")
print(f"  - Improcedência: {len(df_improcedencia)}")

df_decisoes_completo


=== TODOS OS REGISTROS COM DECISÃO ===
Total de registros: 223

Distribuição por tipo:
  - Procedência: 187
  - Improcedência: 36


,numero_processo,tipo_decisao
6,00006577520258272737,Procedência
7,00075235420248272731,Procedência
15,00045760920248272737,Procedência
16,00046049220248272731,Procedência
18,00045234620248272731,Procedência
...,...,...
208,00073810420198272706,Improcedência
218,00215470620188272729,Improcedência
237,00223502920168272706,Improcedência
242,00144266420168272706,Improcedência


## 🧹 6. LIMPEZA E CONSOLIDAÇÃO DO DATASET

### 6.1 Unificação dos DataFrames
**Descrição**: Integração da coluna de tipo de decisão ao DataFrame principal.

In [18]:
df['tipo_decisao'] = df_decisoes_completo['tipo_decisao']
display(df.head(4))

,numero_processo,grau,data_ajuizamento,movimentos,tipo_decisao
0,00063598920268272729,G1,20260210171030,"[{'complementosTabelados': [{'codigo': 2, 'val...",NaN
1,00023463720268272700,G2,20260205173639,"[{'complementosTabelados': [{'codigo': 2, 'val...",NaN
2,00007286620238272731,G2,20260204164448,"[{'complementosTabelados': [{'codigo': 2, 'val...",Improcedência
3,00037157620268272729,G1,20260128120123,"[{'complementosTabelados': [{'codigo': 2, 'val...",NaN


### 6.2 Remoção de Valores Nulos

In [19]:
# Removendo registros NaN
df = df.dropna(subset=['tipo_decisao'])
display(df.head(4))

,numero_processo,grau,data_ajuizamento,movimentos,tipo_decisao
2,00007286620238272731,G2,20260204164448,"[{'complementosTabelados': [{'codigo': 2, 'val...",Improcedência
6,00008591420268272706,G1,20260116221346,"[{'complementosTabelados': [{'codigo': 2, 'val...",Procedência
7,00007795020268272706,G1,20260115173242,"[{'complementosTabelados': [{'codigo': 2, 'val...",Procedência
8,00001690720268272731,G1,20260114145715,"[{'complementosTabelados': [{'codigo': 2, 'val...",Improcedência


### 6.3 Remoção de Colunas Intermediárias

In [20]:
# Removendo colunas intermediárias
df = df.drop(columns=['grau', 'movimentos', 'tipo_decisao'])
display(df.head(4))

,numero_processo,data_ajuizamento
2,00007286620238272731,20260204164448
6,00008591420268272706,20260116221346
7,00007795020268272706,20260115173242
8,00001690720268272731,20260114145715


### 6.4 Reset de Índice

In [21]:
# Resetando index para manter organização do dataframe
df = df.reset_index(drop=True)
display(df.head(4))

,numero_processo,data_ajuizamento
0,00007286620238272731,20260204164448
1,00008591420268272706,20260116221346
2,00007795020268272706,20260115173242
3,00001690720268272731,20260114145715


## 📅 7. EXTRAÇÃO DE FEATURES TEMPORAIS

### 7.1 Extração de Dia da Semana e Ano
**Descrição**: Criação de features temporais (dia da semana e ano) para análise de padrões temporais nas decisões judiciais.

In [22]:
import pandas as pd

# Converter data_ajuizamento para datetime
df['data_ajuizamento_dt'] = pd.to_datetime(
    df['data_ajuizamento'],
    format='%Y%m%d%H%M%S'
)

dias_semana = {0: "Segunda", 1: "Terça", 2: "Quarta", 3: "Quinta", 4: "Sexta"}

# Extrair dia da semana (segunda = 0 | terça = 1 | ... | sexta = 4)
df['dia_semana'] = df['data_ajuizamento_dt'].dt.weekday.map(dias_semana)
df['ano'] = df['data_ajuizamento_dt'].dt.year

# Excluir coluna intermediária
df = df.drop(columns=['data_ajuizamento', 'data_ajuizamento_dt'])

df

,numero_processo,dia_semana,ano
0,00007286620238272731,Quarta,2026
1,00008591420268272706,Sexta,2026
2,00007795020268272706,Quinta,2026
3,00001690720268272731,Quarta,2026
4,00001872420268272700,Segunda,2026
...,...,...,...
218,00141108820248272700,Quarta,2024
219,00049045420248272731,Quarta,2024
220,00140952220248272700,Quarta,2024
221,00028343920248272707,Segunda,2024


### 7.2 Verificando ano mínimo e máximo do conjunto de dados

In [23]:
min = df['ano'].min()
max = df['ano'].max()

print(f"Ano mínimo: {min}")
print(f"Ano máximo: {max}")

Ano mínimo: 2024
Ano máximo: 2026


## 💾 8. EXPORTAÇÃO DE DADOS INTERMEDIÁRIOS

### 8.1 Exportação para CSV
**Descrição**: Exportação dos dados processados para uso posterior:

*  processos_decisao_dia_ano.csv: Dataset completo com decisões e features temporais
*  numeros_processos.csv: Lista única de números de processo para scraping de PDFs




In [24]:
df.to_csv("processos_dia_ano.csv", sep=';', index=False)
df['numero_processo'].drop_duplicates().to_csv("numeros_processos.csv", index=False)

print("\n=== ARQUIVOS EXPORTADOS ===")
print("  - processos_dia_ano.csv")
print("  - numeros_processos.csv")


=== ARQUIVOS EXPORTADOS ===
  - processos_dia_ano.csv
  - numeros_processos.csv


## 📄 9. INSTRUÇÕES PARA COLETA DE DOCUMENTOS COMPLETOS

### 9.1 Scraping de PDFs (Executar Localmente)
**Pré-requisitos:**
1. Clone o repositório: `git clone https://github.com/GiovanniBrigido/trabalho-final-deep-learning.git`
2. Navegue até o diretório do projeto
3. Execute os seguintes comandos:

**Comandos:**
```bash
python -m venv venv
pip install -r requirements.txt
playwright install chromium
python ./scraper_pdf_tjce.py
```

O que o script faz:

*   Lê o arquivo numeros_processos.csv
*   Acessa o site do TJCE
*   Baixa os PDFs das sentenças completas
*   Salva os arquivos na pasta data/decisoes





### 9.2 Extração de Texto dos PDFs (Executar Localmente)
Após gerar os arquivos básicos acima, execute o script `scraper_pdf_tjce.py` para realizar a extração de PDF's.

**Execute:**
```bash
python ./extrair_decisoes.py
```
**O que o script faz**:

* Lê todos os PDFs baixados
* Extrai o texto completo de cada sentença com a bib PyPDF2
* Gera o arquivo decisoes_extraidas.csv com:
   * Número do processo
   * Texto completo da decisão
   * Metadados adicionais

**Arquivo gerado:**

*  decisoes_extraidas.csv
* **!! SERÁ USADO NO NOTEBOOK 2 !!**